# Esecuzione completa — correzioni del relatore (ottobre 2026)

Branch `fix/correzioni-relatore-ottobre`. Il notebook ha quattro fasi, da lanciare **in quest'ordine**:

- **A. Dati esterni** (una sola volta): snapshot Open/High/Low/Close e serie realmente rollata dai contratti singoli. I file vanno poi aggiunti al branch, così dati, codice e risultati stanno nello stesso commit.
- **B. Esecuzione principale** (`python main.py`, seed 42, circa 5–6 ore su GPU).
- **C. Esecuzioni di robustezza**: 4 seed aggiuntivi per le reti e TCN-31. Una cella per sessione, **dallo stesso commit** della fase B.
- **D. Riepilogo e zip** dei risultati.

Le scelte metodologiche sono fissate in `CODE/PIANO_CORREZIONI.md` e registrate in `run_manifest.json`: non vanno cambiate dopo aver visto i risultati.

## 0. Preparazione: GPU, Drive, repository, dipendenze

In [ ]:
!nvidia-smi

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponibili per TensorFlow: {gpus}")
if not gpus:
    print("ATTENZIONE: nessuna GPU rilevata — Runtime > Change runtime type > GPU.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/niccolozoboli/Quantitative-Finance-Master-Thesis.git"
BRANCH   = "fix/correzioni-relatore-ottobre"
REPO_DIR = "/content/Quantitative-Finance-Master-Thesis"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {REPO_DIR}/CODE
COMMIT = !git rev-parse HEAD
COMMIT = COMMIT[0]
print("Commit:", COMMIT)

In [ ]:
!pip install -q -r requirements.txt

RESULTS_DIR = "/content/drive/MyDrive/thesis_results/correzioni-ottobre"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["RESULTS_DIR"] = RESULTS_DIR
print(f"RESULTS_DIR = {RESULTS_DIR}")

## A. Dati esterni (una sola volta, poi aggiungerli al branch)

1. Le due celle sotto scaricano gli snapshot e stampano l'esito dei controlli.
2. La terza copia i file nuovi su Drive (`thesis_results/dati_esterni/`).
3. **Inviali a Claude** (o caricali tu nel branch, nella cartella `CODE/data/`): verranno aggiunti con un commit.
4. Solo dopo, riparti dalla cella 0 (nuovo commit) e passa alla fase B.

Se un controllo fallisce, il file non va aggiunto: la pipeline userà le alternative previste (ritardo di un giorno, sensitivity sui giorni di roll) e lo registrerà nel manifest.

In [ ]:
!python tools/fetch_ohlc.py

In [ ]:
!python tools/build_rolled_series.py

In [ ]:
import glob, shutil
DATA_OUT = "/content/drive/MyDrive/thesis_results/dati_esterni"
os.makedirs(DATA_OUT, exist_ok=True)
for f in glob.glob("data/*_ohlc.csv") + glob.glob("data/*_rolled.csv"):
    shutil.copy(f, DATA_OUT)
    print("copiato", f)

## B. Esecuzione principale — `python main.py`

Mai `--resume` al primo giro: `--resume` serve solo a riprendere un'esecuzione **di questo stesso commit** interrotta a metà (vedi `_check_resume_commit`). La cella controlla prima che gli snapshot esterni attesi siano presenti nel commit.

In [ ]:
import glob
print("Snapshot OHLC nel commit:", sorted(glob.glob("data/*_ohlc.csv")))
print("Serie rollate nel commit:", sorted(glob.glob("data/*_rolled.csv")))

In [ ]:
!python main.py

## C. Esecuzioni di robustezza (stesso commit della fase B)

Una cella per sessione Colab se il tempo di sessione non basta. Ogni esecuzione scrive file e manifest propri e non tocca quelli della fase B.

In [ ]:
!python main.py --robustness tcn31

In [ ]:
!python main.py --dl-seed 1

In [ ]:
!python main.py --dl-seed 2

In [ ]:
!python main.py --dl-seed 3

In [ ]:
!python main.py --dl-seed 4

## D. Riepilogo della robustezza e zip dei risultati

In [ ]:
!python tools/summarize_robustness.py

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = f"/content/drive/MyDrive/thesis_results/results_correzioni_{stamp}.zip"
!zip -r -q {zip_path} {RESULTS_DIR}
print(f"Risultati compressi in: {zip_path}")